# TERA — Multilingual PDF Research

Compare factual extraction from English and Chinese synthetic hotel invoices, then generate a German README-style expense summary. PDF text is extracted directly.

The Pydantic schema is an evaluation instrument only. It is passed via Ollama's structured-output parameter for the extraction experiment, never inserted into prompts. The summary uses document context directly, without a schema.


In [1]:
import pymupdf
import json
import requests
import pandas as pd
import time
from pydantic import BaseModel
from pathlib import Path
import sys

RESEARCH_DIR = Path.cwd() if (Path.cwd() / "prompts.py").is_file() else Path.cwd() / "research"
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))
from prompts import build_extraction_prompt, build_summary_prompt
from IPython.display import display, Markdown


## 1. Research-only evaluation schema

Field accuracy measures model extraction from PDF text, not PDF rendering quality. Original names and payment wording should be preserved. Room rate means the stated nightly room price; breakfast amount is the full-stay gross breakfast charge in the reference data. Strict matches can penalize equivalent wording.

In [2]:
class HotelInvoiceExtraction(BaseModel):
    hotel_name: str | None
    hotel_address: str | None
    invoice_number: str | None
    invoice_date: str | None
    guest_name: str | None
    check_in_date: str | None
    check_out_date: str | None
    number_of_nights: int | None
    currency: str | None
    room_rate: float | None
    city_tax: float | None
    total_amount: float | None
    payment_method: str | None
    breakfast_amount: float | None


## 2. Fixtures and manually verified ground truth

In [3]:
DATA_DIR = RESEARCH_DIR / "data"
GROUND_TRUTH = {'hotel_invoice.pdf': {'hotel_name': 'NORTHSTAR HOTEL BERLIN',
                       'hotel_address': 'Alexanderplatz 99, 10178 Berlin, Germany',
                       'invoice_number': 'NSB-2026-0914-1042',
                       'invoice_date': '2026-09-14',
                       'guest_name': 'Alex Morgan',
                       'check_in_date': '2026-09-10',
                       'check_out_date': '2026-09-14',
                       'number_of_nights': 4,
                       'currency': 'EUR',
                       'room_rate': 140.0,
                       'city_tax': 42.0,
                       'total_amount': 712.6,
                       'payment_method': 'Visa ending 4242',
                       'breakfast_amount': 71.4},
 'hotel_invoice_zh.pdf': {'hotel_name': '上海晨星酒店',
                          'hotel_address': '上海市浦东新区示例路88号',
                          'invoice_number': 'CX-2026-0918-2048',
                          'invoice_date': '2026-09-18',
                          'guest_name': '李明',
                          'check_in_date': '2026-09-15',
                          'check_out_date': '2026-09-18',
                          'number_of_nights': 3,
                          'currency': 'CNY',
                          'room_rate': 680.0,
                          'city_tax': 0.0,
                          'total_amount': 2220.0,
                          'payment_method': '支付宝',
                          'breakfast_amount': 180.0}}


## 3. Extract and inspect PDF text

Document names and page numbers remain attached to the text. Empty pages stop the experiment.

In [4]:
documents = []
for filename in GROUND_TRUTH:
    with pymupdf.open(DATA_DIR / filename) as pdf:
        pages = [{"page": p.number + 1, "text": p.get_text()} for p in pdf]
    if not pages or any(not p["text"].strip() for p in pages):
        raise ValueError(f"{filename}: missing extractable text")
    documents.append({"document_id": filename, "pages": pages})
    print(filename)
    print("\n".join(p["text"] for p in pages))

chinese_text = "\n".join(p["text"] for p in documents[1]["pages"])
for expected in ("上海晨星酒店", "李明", "早餐", "2220.00", "支付宝"):
    assert expected in chinese_text, f"Missing Chinese PDF text: {expected}"


hotel_invoice.pdf
SYNTHETIC TEST INVOICE | Machine-readable PDF with selectable text
Page 1
NORTHSTAR HOTEL BERLIN
Fictional property - synthetic test document
Alexanderplatz 99
10178 Berlin, Germany
+49 30 5550 0199
billing@northstar-example.test
INVOICE
PAID
INVOICE NUMBER
NSB-2026-0914-1042
INVOICE DATE
14 September 2026
GUEST NAME
Alex Morgan
BOOKING REFERENCE
TERA-DEMO-74291
CHECK-IN
10 September 2026, 15:00
CHECK-OUT
14 September 2026, 11:00
STAY
4 room nights
ROOM
Deluxe King - 508
CHARGES
Description
Qty
Unit rate
Net
Tax
Amount
Accommodation - Deluxe King
4 nights
EUR 140.00
EUR 560.00
VAT 7%
EUR 599.20
Breakfast
4
EUR 15.00
EUR 60.00
VAT 19%
EUR 71.40
Berlin city tax (7.5% of room net)
1
EUR 42.00
EUR 42.00
Exempt
EUR 42.00
Tax summary
Taxable base
Tax amount
VAT 7% - accommodation
EUR 560.00
EUR 39.20
VAT 19% - breakfast
EUR 60.00
EUR 11.40
City tax - exempt from VAT
EUR 42.00
EUR 0.00
Subtotal (net + city tax)
EUR 662.00
VAT total
EUR 50.60
TOTAL AMOUNT
EUR 712.60
Currency


## 4. Model requests

Only the research extraction call supplies `format`. The summary request has no output schema. Calls remain local to Ollama; a failed model is reported separately from accuracy.

In [5]:
MODELS = ["gemma3:12b", "qwen3:14b", "qwen3.5:9b"]

def call_model(model_name: str, prompt: str, research_schema=None) -> str:
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0, "num_ctx": 8192},
    }
    if research_schema is not None:
        payload["format"] = research_schema
    if model_name.startswith("qwen3"):
        payload["think"] = False
    response = requests.post(
        "http://localhost:11434/api/generate", json=payload, timeout=600
    )
    response.raise_for_status()
    result = response.json()
    if not result.get("done") or result.get("done_reason") == "length":
        raise ValueError("Model response was truncated")
    return result["response"]


In [6]:
def parse_model_output(output: str) -> dict:
    cleaned = output.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):]

    elif cleaned.startswith("```"):
        cleaned = cleaned[len("```"):]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    return json.loads(cleaned)


def validate_extracted_data(extracted_data: dict) -> HotelInvoiceExtraction:
    return HotelInvoiceExtraction.model_validate(extracted_data)

## 5. Run extraction on both languages

In [7]:
results = []
for model_name in MODELS:
    for document in documents:
        filename = document["document_id"]
        print(f"Running {model_name}: {filename}...", flush=True)
        start = time.perf_counter()
        raw_output = None
        try:
            raw_output = call_model(
                model_name, build_extraction_prompt([document]),
                research_schema=HotelInvoiceExtraction.model_json_schema(),
            )
            extracted = validate_extracted_data(parse_model_output(raw_output)).model_dump()
            results.append({"model": model_name, "document": filename,
                            "data": extracted, "raw_output": raw_output,
                            "seconds": time.perf_counter() - start, "error": None})
            print(json.dumps(extracted, ensure_ascii=False, indent=2))
        except Exception as exc:
            results.append({"model": model_name, "document": filename,
                            "data": None, "raw_output": raw_output,
                            "seconds": time.perf_counter() - start, "error": str(exc)})
            print(f"FAILED: {exc}")


Running gemma3:12b: hotel_invoice.pdf...
{
  "hotel_name": "NORTHSTAR HOTEL BERLIN",
  "hotel_address": "Alexanderplatz 99, 10178 Berlin, Germany",
  "invoice_number": "NSB-2026-0914-1042",
  "invoice_date": "2026-09-14",
  "guest_name": "Alex Morgan",
  "check_in_date": "2026-09-10",
  "check_out_date": "2026-09-14",
  "number_of_nights": 4,
  "currency": "EUR",
  "room_rate": 140.0,
  "city_tax": 42.0,
  "total_amount": 712.6,
  "payment_method": "Visa ending 4242",
  "breakfast_amount": 60.0
}
Running gemma3:12b: hotel_invoice_zh.pdf...
{
  "hotel_name": "上海晨星酒店",
  "hotel_address": "上海市浦东新区示例路88号",
  "invoice_number": "CX-2026-0918-2048",
  "invoice_date": "2026-09-18",
  "guest_name": "李明",
  "check_in_date": "2026-09-15",
  "check_out_date": "2026-09-18",
  "number_of_nights": 3,
  "currency": "CNY",
  "room_rate": 680.0,
  "city_tax": 0.0,
  "total_amount": 2220.0,
  "payment_method": "支付宝",
  "breakfast_amount": 180.0
}
Running qwen3:14b: hotel_invoice.pdf...
{
  "hotel_name": 

## 6. Field-level accuracy and errors

Ground truth is used only after generation. Failed runs have no accuracy score. These two synthetic invoices do not establish general multilingual accuracy.

In [8]:
evaluation_rows = []
for result in results:
    matches = None
    if result["error"] is None:
        expected = GROUND_TRUTH[result["document"]]
        comparison = pd.DataFrame([
            {"field": field, "expected": value, "actual": result["data"].get(field),
             "match": result["data"].get(field) == value}
            for field, value in expected.items()
        ])
        matches = int(comparison["match"].sum())
        print(result["model"], result["document"])
        display(comparison)
    evaluation_rows.append({
        "model": result["model"], "document": result["document"],
        "correct_fields": matches, "total_fields": len(GROUND_TRUTH[result["document"]]),
        "accuracy": matches / len(GROUND_TRUTH[result["document"]]) if matches is not None else None,
        "seconds": round(result["seconds"], 2), "error": result["error"],
    })
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)


gemma3:12b hotel_invoice.pdf
gemma3:12b hotel_invoice_zh.pdf
qwen3:14b hotel_invoice.pdf
qwen3:14b hotel_invoice_zh.pdf
qwen3.5:9b hotel_invoice.pdf
qwen3.5:9b hotel_invoice_zh.pdf


,field,expected,actual,match
0,hotel_name,NORTHSTAR HOTEL BERLIN,NORTHSTAR HOTEL BERLIN,True
1,hotel_address,"Alexanderplatz 99, 10178 Berlin, Germany","Alexanderplatz 99, 10178 Berlin, Germany",True
2,invoice_number,NSB-2026-0914-1042,NSB-2026-0914-1042,True
3,invoice_date,2026-09-14,2026-09-14,True
4,guest_name,Alex Morgan,Alex Morgan,True
5,check_in_date,2026-09-10,2026-09-10,True
6,check_out_date,2026-09-14,2026-09-14,True
7,number_of_nights,4,4,True
8,currency,EUR,EUR,True
9,room_rate,140.0,140.0,True


,field,expected,actual,match
0,hotel_name,上海晨星酒店,上海晨星酒店,True
1,hotel_address,上海市浦东新区示例路88号,上海市浦东新区示例路88号,True
2,invoice_number,CX-2026-0918-2048,CX-2026-0918-2048,True
3,invoice_date,2026-09-18,2026-09-18,True
4,guest_name,李明,李明,True
5,check_in_date,2026-09-15,2026-09-15,True
6,check_out_date,2026-09-18,2026-09-18,True
7,number_of_nights,3,3,True
8,currency,CNY,CNY,True
9,room_rate,680.0,680.0,True


,field,expected,actual,match
0,hotel_name,NORTHSTAR HOTEL BERLIN,NORTHSTAR HOTEL BERLIN,True
1,hotel_address,"Alexanderplatz 99, 10178 Berlin, Germany","Alexanderplatz 99, 10178 Berlin, Germany",True
2,invoice_number,NSB-2026-0914-1042,NSB-2026-0914-1042,True
3,invoice_date,2026-09-14,2026-09-14,True
4,guest_name,Alex Morgan,Alex Morgan,True
5,check_in_date,2026-09-10,2026-09-10,True
6,check_out_date,2026-09-14,2026-09-14,True
7,number_of_nights,4,4,True
8,currency,EUR,EUR,True
9,room_rate,140.0,140.0,True


,field,expected,actual,match
0,hotel_name,上海晨星酒店,上海晨星酒店,True
1,hotel_address,上海市浦东新区示例路88号,上海市浦东新区示例路88号,True
2,invoice_number,CX-2026-0918-2048,CX-2026-0918-2048,True
3,invoice_date,2026-09-18,2026-09-18,True
4,guest_name,李明,李明,True
5,check_in_date,2026-09-15,2026-09-15,True
6,check_out_date,2026-09-18,2026-09-18,True
7,number_of_nights,3,3,True
8,currency,CNY,CNY,True
9,room_rate,680.0,680.0,True


,field,expected,actual,match
0,hotel_name,NORTHSTAR HOTEL BERLIN,NORTHSTAR HOTEL BERLIN,True
1,hotel_address,"Alexanderplatz 99, 10178 Berlin, Germany","Alexanderplatz 99\n10178 Berlin, Germany",False
2,invoice_number,NSB-2026-0914-1042,NSB-2026-0914-1042,True
3,invoice_date,2026-09-14,2026-09-14,True
4,guest_name,Alex Morgan,Alex Morgan,True
5,check_in_date,2026-09-10,2026-09-10,True
6,check_out_date,2026-09-14,2026-09-14,True
7,number_of_nights,4,4,True
8,currency,EUR,EUR,True
9,room_rate,140.0,140.0,True


,field,expected,actual,match
0,hotel_name,上海晨星酒店,上海晨星酒店,True
1,hotel_address,上海市浦东新区示例路88号,上海市浦东新区示例路88号,True
2,invoice_number,CX-2026-0918-2048,CX-2026-0918-2048,True
3,invoice_date,2026-09-18,2026-09-18,True
4,guest_name,李明,李明,True
5,check_in_date,2026-09-15,2026-09-15,True
6,check_out_date,2026-09-18,2026-09-18,True
7,number_of_nights,3,3,True
8,currency,CNY,CNY,True
9,room_rate,680.0,680.0,True


,model,document,correct_fields,total_fields,accuracy,seconds,error
0,gemma3:12b,hotel_invoice.pdf,13,14,0.928571,18.64,None
1,gemma3:12b,hotel_invoice_zh.pdf,14,14,1.000000,10.49,None
2,qwen3:14b,hotel_invoice.pdf,13,14,0.928571,13.49,None
3,qwen3:14b,hotel_invoice_zh.pdf,14,14,1.000000,8.92,None
4,qwen3.5:9b,hotel_invoice.pdf,12,14,0.857143,11.23,None
5,qwen3.5:9b,hotel_invoice_zh.pdf,14,14,1.000000,6.07,None


## 7. German Markdown summary without a schema

Generate a combined report directly from both PDF contexts. Export successful reports to `research/results/summary_<model>_de.md`. Review arithmetic and sources before using any report.

In [9]:
OUTPUT_DIR = RESEARCH_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)
from datetime import datetime, timezone
from hashlib import sha256

run_metadata = {
    "completed_at_utc": None,
    "models": MODELS,
    "temperature": 0,
    "context_tokens": 8192,
    "prompt_sha256": sha256((RESEARCH_DIR / "prompts.py").read_bytes()).hexdigest(),
    "template_sha256": sha256((RESEARCH_DIR / "summary_template.md").read_bytes()).hexdigest(),
    "documents_sha256": {
        name: sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in GROUND_TRUTH
    },
}
summary_results = {}
for model_name in MODELS:
    print(f"Generating German summary: {model_name}...", flush=True)
    output_path = OUTPUT_DIR / f"summary_{model_name.replace(':', '_')}_de.md"
    try:
        summary = call_model(model_name, build_summary_prompt(documents))
        output_path.write_text(summary, encoding="utf-8")
        summary_results[model_name] = {"path": output_path.name, "error": None}
        display(Markdown(summary))
    except Exception as exc:
        summary_results[model_name] = {"path": None, "error": str(exc)}
        print(f"FAILED: {exc}")

run_metadata["completed_at_utc"] = datetime.now(timezone.utc).isoformat()

(OUTPUT_DIR / "evaluation.json").write_text(
    json.dumps({"run": run_metadata, "extraction": results, "evaluation": evaluation_rows,
                "summaries": summary_results}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


Generating German summary: gemma3:12b...
Generating German summary: qwen3:14b...
Generating German summary: qwen3.5:9b...
Out[9]: 9334


# Reisekostenübersicht

## Belegübersicht

- Bereitgestellt: 2
- Verarbeitet: 2
- Zu prüfen: 0

Nur bereitgestellte Dokumente berücksichtigt. Summen sind vorläufig.

## Ausgaben nach Datum

| Datum | Leistungszeitraum | Anbieter | Kategorie | Beschreibung | Betrag | Währung | Beleg / Seite |
| --- | --- | --- | --- | --- | ---: | --- | --- |
| 2026-09-14 | 2026-09-10 bis 2026-09-14 | NORTHSTAR HOTEL BERLIN | Hotel | Accommodation - Deluxe King | 560.00 | EUR | hotel_invoice.pdf / 1 |
| 2026-09-14 | 2026-09-10 bis 2026-09-14 | NORTHSTAR HOTEL BERLIN | Verpflegung | Breakfast | 60.00 | EUR | hotel_invoice.pdf / 1 |
| 2026-09-14 | 2026-09-10 bis 2026-09-14 | NORTHSTAR HOTEL BERLIN | Sonstige Ausgaben | Berlin city tax (7.5% of room net) | 42.00 | EUR | hotel_invoice.pdf / 1 |
| 2026-09-18 | 2026-09-15 bis 2026-09-18 | 上海晨星酒店 | Hotel | 住宿费 | 2040.00 | CNY | hotel_invoice_zh.pdf / 1 |
| 2026-09-18 | 2026-09-15 bis 2026-09-18 | 上海晨星酒店 | Verpflegung | 早餐费 | 180.00 | CNY | hotel_invoice_zh.pdf / 1 |
| 2026-09-18 | 2026-09-15 bis 2026-09-18 | 上海晨星酒店 | Sonstige Ausgaben | 城市税 | 0.00 | CNY | hotel_invoice_zh.pdf / 1 |

## Tagessummen

| Datum | Währung | Bestätigt | In Prüfung |
| --- | --- | ---: | ---: |
| 2026-09-14 | EUR | 712.60 | 0.00 |
| 2026-09-18 | CNY | 2220.00 | 0.00 |

## Summen nach Kategorie

| Kategorie | Währung | Bestätigt | In Prüfung |
| --- | --- | ---: | ---: |
| Hotel | EUR | 602.00 | 0.00 |
| Flugreisen | EUR | 0.00 | 0.00 |
| Verpflegung | EUR | 60.00 | 0.00 |
| Sonstige Ausgaben | EUR | 42.00 | 0.00 |
| Hotel | CNY | 2040.00 | 0.00 |
| Flugreisen | CNY | 0.00 | 0.00 |
| Verpflegung | CNY | 180.00 | 0.00 |
| Sonstige Ausgaben | CNY | 0.00 | 0.00 |

## Gesamtsummen

| Währung | Bestätigt | In Prüfung |
| --- | ---: | ---: |
| EUR | 712.60 | 0.00 |
| CNY | 2220.00 | 0.00 |

## Unterkunft und Frühstück

| Beleg / Seite | Zeitraum | Währung | Mit Frühstück | Frühstück | Ohne Frühstück | Berechnung / Hinweise |
| --- | --- | --- | ---: | ---: | ---: | --- |
| hotel_invoice.pdf / 1 | 2026-09-10 bis 2026-09-14 | EUR | Ja | 60.00 | Unbekannt | Gesamter Aufenthalt, Frühstück inklusive. |
| hotel_invoice_zh.pdf / 1 | 2026-09-15 bis 2026-09-18 | CNY | Ja | 180.00 | Unbekannt | Gesamter Aufenthalt, Frühstück inklusive. |

## Prüfhinweise

- hotel_invoice.pdf / 1: Diskrepanz zwischen Gesamtbetrag (712.60 EUR) und Summe der einzelnen Ausgaben (602.00 EUR + 60.00 EUR + 42.00 EUR = 704.00 EUR). Differenz: 8.60 EUR. Ursache unklar.
- hotel_invoice_zh.pdf / 1: Der Gesamtbetrag (2220.00 CNY) beinhaltet sowohl die Unterkunft als auch das Frühstück. Der Betrag ohne Frühstück (2040.00 CNY) wurde als Unterkunft ausgewiesen.
- hotel_invoice_zh.pdf / 1: Die Beschreibung "城市税" (city tax) ist unklar. Es ist unklar, ob es sich um eine lokale Steuer handelt oder ob sie in den anderen Ausgaben enthalten ist.
- hotel_invoice_zh.pdf / 1: Die Angaben zum Mehrwertsteueranteil (6%) sind unvollständig.
Keine Einträge.

# Reisekostenübersicht

## Belegübersicht

- Bereitgestellt: 2
- Verarbeitet: 2
- Zu prüfen: 0

Nur bereitgestellte Dokumente berücksichtigt. Summen sind vorläufig.

## Ausgaben nach Datum

| Datum | Leistungszeitraum | Anbieter | Kategorie | Beschreibung | Betrag | Währung | Beleg / Seite |
| --- | --- | --- | --- | --- | ---: | --- | --- |
| 2026-09-14 | 10.09.2026 – 14.09.2026 | NORTHSTAR HOTEL BERLIN | Hotel | Unterkunft (Deluxe King) | 599,20 | EUR | hotel_invoice.pdf / 1 |
| 2026-09-14 | 10.09.2026 – 14.09.2026 | NORTHSTAR HOTEL BERLIN | Verpflegung | Frühstück | 71,40 | EUR | hotel_invoice.pdf / 1 |
| 2026-09-18 | 15.09.2026 – 18.09.2026 | 上海晨星酒店 | Hotel | Unterkunft (Luxus-Doppelzimmer) | 2040,00 | CNY | hotel_invoice_zh.pdf / 1 |
| 2026-09-18 | 15.09.2026 – 18.09.2026 | 上海晨星酒店 | Verpflegung | Frühstück | 180,00 | CNY | hotel_invoice_zh.pdf / 1 |

## Tagessummen

| Datum | Währung | Bestätigt | In Prüfung |
| --- | --- | ---: | ---: |
| 2026-09-14 | EUR | 670,60 | 0,00 |
| 2026-09-18 | CNY | 2220,00 | 0,00 |

## Summen nach Kategorie

| Kategorie | Währung | Bestätigt | In Prüfung |
| --- | --- | ---: | ---: |
| Hotel | EUR | 599,20 | 0,00 |
| Verpflegung | EUR | 71,40 | 0,00 |
| Hotel | CNY | 2040,00 | 0,00 |
| Verpflegung | CNY | 180,00 | 0,00 |

## Gesamtsummen

| Währung | Bestätigt | In Prüfung |
| --- | ---: | ---: |
| EUR | 670,60 | 0,00 |
| CNY | 2220,00 | 0,00 |

## Unterkunft und Frühstück

| Beleg / Seite | Zeitraum | Währung | Mit Frühstück | Frühstück | Ohne Frühstück | Berechnung / Hinweise |
| --- | --- | --- | ---: | ---: | ---: | --- |
| hotel_invoice.pdf / 1 | 10.09.2026 – 14.09.2026 | EUR | 712,60 | 71,40 | 641,20 | Frühstück ist separat abgerechnet. |
| hotel_invoice_zh.pdf / 1 | 15.09.2026 – 18.09.2026 | CNY | 2220,00 | 180,00 | 2040,00 | Frühstück ist separat abgerechnet. |

## Prüfhinweise

- Keine.

# Reisekostenübersicht

## Belegübersicht

- Bereitgestellt: 2
- Verarbeitet: 2
- Zu prüfen: 0

Nur bereitgestellte Dokumente berücksichtigt. Summen sind vorläufig.

## Ausgaben nach Datum

| Datum | Leistungszeitraum | Anbieter | Kategorie | Beschreibung | Betrag | Währung | Beleg / Seite |
| --- | --- | --- | --- | --- | ---: | --- | --- |
| 2026-09-14 | 2026-09-10 bis 2026-09-14 | NORTHSTAR HOTEL BERLIN | Hotel | Accommodation - Deluxe King, Berlin city tax | 602.00 | EUR | NSB-2026-0914-1042 / 1 |
| 2026-09-14 | 2026-09-10 bis 2026-09-14 | NORTHSTAR HOTEL BERLIN | Verpflegung | Breakfast, VAT 19% | 71.40 | EUR | NSB-2026-0914-1042 / 1 |
| 2026-09-18 | 2026-09-15 bis 2026-09-18 | 上海晨星酒店 | Hotel | 住宿费，城市税 (0.00), Frühstück (bundled) | 2220.00 | CNY | CX-2026-0918-2048 / 1 |

## Tagessummen

| Datum | Währung | Bestätigt | In Prüfung |
| --- | --- | ---: | ---: |
| 2026-09-14 | EUR | 673.40 | 0.00 |
| 2026-09-18 | CNY | 2220.00 | 0.00 |

## Summen nach Kategorie

| Kategorie | Währung | Bestätigt | In Prüfung |
| --- | --- | ---: | ---: |
| Flugreisen | EUR | 0.00 | 0.00 |
| Flugreisen | CNY | 0.00 | 0.00 |
| Hotel | EUR | 602.00 | 0.00 |
| Hotel | CNY | 2220.00 | 0.00 |
| Sonstige Ausgaben | EUR | 0.00 | 0.00 |
| Sonstige Ausgaben | CNY | 0.00 | 0.00 |
| Verpflegung | EUR | 71.40 | 0.00 |
| Verpflegung | CNY | 180.00 | 0.00 |

## Gesamtsummen

| Währung | Bestätigt | In Prüfung |
| --- | ---: | ---: |
| EUR | 673.40 | 0.00 |
| CNY | 2220.00 | 0.00 |

## Unterkunft und Frühstück

| Beleg / Seite | Zeitraum | Währung | Mit Frühstück | Frühstück | Ohne Frühstück | Berechnung / Hinweise |
| --- | --- | --- | ---: | ---: | ---: | --- |
| NSB-2026-0914-1042 / 1 | 2026-09-10 bis 2026-09-14 | EUR | 712.60 | 71.40 | 641.20 | Differenz enthält MwSt. (VAT) für Unterkunft und Stadtsteuer. |
| CX-2026-0918-2048 / 1 | 2026-09-15 bis 2026-09-18 | CNY | 2220.00 | 180.00 | 2040.00 | Frühstück separat berechnet; Differenz entspricht Netto-Akkommodation ohne Frühstück. |

## Prüfhinweise

- NSB-2026-0914-1042 / 1: Keine.
- CX-2026-0918-2048 / 1: Keine.

## 8. Review checklist

- Check field-level results for both documents; distinguish wording differences from incorrect facts.
- Chinese invoice: CNY 2220.00 total, CNY 180.00 breakfast, CNY 2040.00 without breakfast for the full stay. VAT is already included.
- English invoice: EUR 712.60 total, EUR 71.40 breakfast, EUR 641.20 excluding breakfast **including city tax**. Room-only gross amount is EUR 599.20; the stated nightly net rate is EUR 140.00.
- Keep EUR and CNY totals separate and retain source references.
- Verify that generated summaries use German headings, prose, and tables. Original merchant names stay unchanged.
- Model output is experimental; larger and more varied datasets are needed.


### Summary review

See [results/README.md](results/README.md) for the manual review of the saved model summaries. The tables above show extraction results from this execution.
